In [ ]:
pip install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121


In [1]:
import torch
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True, num_workers=2)

100%|██████████| 170M/170M [04:00<00:00, 709kB/s]


In [2]:
import torch.nn as nn
import torch.nn.functional as F

class ConvHighwayBlock(nn.Module):
    def __init__(self, in_channels):
        super(ConvHighwayBlock, self).__init__()
        self.H = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1)
        self.T = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1)
        self.T.bias.data.fill_(-2.0)

    def forward(self, x):
        h = F.relu(self.H(x))
        t = torch.sigmoid(self.T(x))
        c = 1.0 - t
        y = h * t + x * c
        return y

class CIFAR10HighwayNet(nn.Module):
    def __init__(self, num_layers=10, num_classes=10):
        super(CIFAR10HighwayNet, self).__init__()
        self.initial_conv = nn.Conv2d(3, 32, kernel_size=3, padding=1)

        highway_layers = [ConvHighwayBlock(32) for _ in range(num_layers)]
        self.highways = nn.Sequential(*highway_layers)

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(32, num_classes)

    def forward(self, x):
        x = F.relu(self.initial_conv(x))
        x = self.highways(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

In [4]:
import torch.optim as optim
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = CIFAR10HighwayNet(num_layers=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

epochs = 100
for epoch in range(epochs):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch [{epoch + 1}/{epochs}], Loss: {running_loss / len(trainloader):.4f}")

Epoch [1/100], Loss: 2.2445
Epoch [2/100], Loss: 2.0757
Epoch [3/100], Loss: 1.9499
Epoch [4/100], Loss: 1.8336
Epoch [5/100], Loss: 1.7284
Epoch [6/100], Loss: 1.6649
Epoch [7/100], Loss: 1.6140
Epoch [8/100], Loss: 1.5697
Epoch [9/100], Loss: 1.5096
Epoch [10/100], Loss: 1.4507
Epoch [11/100], Loss: 1.3834
Epoch [12/100], Loss: 1.3182
Epoch [13/100], Loss: 1.2482
Epoch [14/100], Loss: 1.2028
Epoch [15/100], Loss: 1.1542
Epoch [16/100], Loss: 1.1081
Epoch [17/100], Loss: 1.0738
Epoch [18/100], Loss: 1.0478
Epoch [19/100], Loss: 1.0246
Epoch [20/100], Loss: 0.9761
Epoch [21/100], Loss: 0.9623
Epoch [22/100], Loss: 0.9373
Epoch [23/100], Loss: 0.9179
Epoch [24/100], Loss: 0.8877
Epoch [25/100], Loss: 0.8699
Epoch [26/100], Loss: 0.8598
Epoch [27/100], Loss: 0.8320
Epoch [28/100], Loss: 0.8233
Epoch [29/100], Loss: 0.8041
Epoch [30/100], Loss: 0.7941
Epoch [31/100], Loss: 0.7898
Epoch [32/100], Loss: 0.7745
Epoch [33/100], Loss: 0.7585
Epoch [34/100], Loss: 0.7484
Epoch [35/100], Loss: 0

In [5]:
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False, num_workers=2)

model.eval()
correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data[0].to(device), data[1].to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy: {accuracy:.2f}%')

Accuracy: 74.61%
